# Global H&E–mIF registration


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Imports


In [ ]:
import gc
import json
import math
import os
import re
import time
import traceback
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional
QPTIFF_CACHE_ROOT = ROOT / '.cache' / 'registration'
QPTIFF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(QPTIFF_CACHE_ROOT / 'huggingface'))
os.environ.setdefault('HF_HUB_CACHE', str(QPTIFF_CACHE_ROOT / 'huggingface' / 'hub'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(QPTIFF_CACHE_ROOT / 'huggingface' / 'transformers'))
os.environ.setdefault('TORCH_HOME', str(QPTIFF_CACHE_ROOT / 'torch'))
import cv2
import numpy as np
import pandas as pd
import tifffile
import torch
import torchvision
import transformers
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModelForKeypointMatching
Image.MAX_IMAGE_PIXELS = None
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Configuration


In [ ]:
@dataclass
class Config:
    HE_DIR: str = str(PATHS.he_dir)
    MIF_DIR: str = str(PATHS.mif_dir)
    OUTPUT_DIR: str = str(PATHS.registration_global)
    EXTENSIONS: tuple[str, ...] = ('.qptiff', '.qptif')
    CASE_ID_REGEX: str = '^(?P<case_id>.+?)_Scan\\d+$'
    RUN_CASE_IDS: tuple[str, ...] = ()
    PYRAMID_TARGET_LONG_EDGE: int = 3000
    PYRAMID_MAX_LONG_EDGE: int = 3500
    MODEL_ID: str = 'zju-community/matchanything_eloftr'
    MODEL_CANVAS: int = 1024
    MATCH_MIN_SCORE: float = 0.1
    SCORE_THRESHOLDS: tuple[float, ...] = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
    USE_MIXED_PRECISION: bool = True
    USE_BIDIRECTIONAL_MATCHING: bool = True
    MUTUAL_TOLERANCE_CANVAS_PX: float = 12.0
    MIN_MUTUAL_MATCHES: int = 12
    GRID_SIZE: int = 8
    MAX_MATCHES_PER_GRID_CELL: int = 24
    MAX_RANSAC_CANDIDATES: int = 1500
    ENABLE_ORIENTATION_RESCUE: bool = False
    RESCUE_ORIENTATIONS: tuple[str, ...] = ('rot90', 'rot180', 'rot270', 'flip_h', 'flip_v')
    RANSAC_THRESHOLD_PX: float = 5.0
    RANSAC_CONFIDENCE: float = 0.999
    RANSAC_MAX_ITERS: int = 30000
    ALLOW_AFFINE: bool = True
    MIN_SCALE: float = 0.25
    MAX_SCALE: float = 4.0
    MAX_ANISOTROPY: float = 1.35
    MAX_TRANSLATION_FRACTION: float = 1.5
    ALLOW_REFLECTION: bool = False
    HARD_MIN_INLIERS: int = 12
    FAIL_MIN_INLIER_RATIO: float = 0.1
    FAIL_MIN_SPATIAL_COVERAGE: float = 0.005
    FAST_ACCEPT_MIN_INLIERS: int = 40
    FAST_ACCEPT_MIN_RATIO: float = 0.5
    FAST_ACCEPT_MIN_COVERAGE: float = 0.05
    QC_MIN_CANDIDATE_MATCHES: int = 30
    QC_MIN_INLIERS: int = 30
    QC_MIN_INLIER_RATIO: float = 0.5
    QC_MIN_SPATIAL_COVERAGE: float = 0.03
    QC_MAX_MEDIAN_ERROR_PX: float = 4.0
    QC_MIN_TISSUE_DICE: float = 0.5
    MAX_QC_MATCHES: int = 250
    RANDOM_SEED: int = 2026
CFG = Config()
HE_DIR = Path(CFG.HE_DIR).resolve()
MIF_DIR = Path(CFG.MIF_DIR).resolve()
OUTPUT_DIR = Path(CFG.OUTPUT_DIR).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'pairs').mkdir(parents=True, exist_ok=True)
assert CFG.MODEL_CANVAS % 32 == 0
assert CFG.MATCH_MIN_SCORE <= min(CFG.SCORE_THRESHOLDS)
assert CFG.QC_MIN_INLIER_RATIO >= 0.5, 'An ok result requires at least as many inliers as outliers.'
assert 'identity' not in CFG.RESCUE_ORIENTATIONS
np.random.seed(CFG.RANDOM_SEED)
torch.manual_seed(CFG.RANDOM_SEED)


## Pyramid reader


In [ ]:
def _yx_shape(shape: tuple[int, ...], axes: str) -> tuple[int, int]:
    axes = axes.upper()
    if 'Y' not in axes or 'X' not in axes:
        raise ValueError(f'Cannot locate Y/X in axes={axes!r}, shape={shape}')
    return (int(shape[axes.index('Y')]), int(shape[axes.index('X')]))

def _select_main_series(tif: tifffile.TiffFile) -> int:
    candidates = []
    for i, series in enumerate(tif.series):
        try:
            h, w = _yx_shape(tuple(series.shape), series.axes)
            candidates.append((h * w, i))
        except Exception:
            continue
    if not candidates:
        raise ValueError('No TIFF series contains X/Y axes')
    return max(candidates)[1]

def inspect_qptiff(path: Path) -> dict[str, Any]:
    with tifffile.TiffFile(path) as tif:
        series_index = _select_main_series(tif)
        series = tif.series[series_index]
        levels = list(getattr(series, 'levels', [series]))
        level_info = []
        for i, level in enumerate(levels):
            h, w = _yx_shape(tuple(level.shape), level.axes)
            level_info.append({'level': i, 'height': h, 'width': w, 'axes': level.axes, 'shape': tuple((int(x) for x in level.shape))})
        return {'path': str(path), 'name': path.name, 'series_index': series_index, 'full_height': level_info[0]['height'], 'full_width': level_info[0]['width'], 'levels': level_info}

def _choose_level(levels: list[dict[str, Any]], target_long_edge: int) -> int:
    target = max(1, int(target_long_edge))
    scores = []
    for info in levels:
        long_edge = max(info['height'], info['width'])
        score = abs(math.log(max(long_edge, 1) / target))
        scores.append((score, info['level']))
    return min(scores)[1]

def _array_to_rgb(arr: np.ndarray, axes: str) -> np.ndarray:
    a = np.asarray(arr)
    ax = list(axes.upper())
    for i in range(a.ndim - 1, -1, -1):
        if a.shape[i] == 1:
            a = np.take(a, 0, axis=i)
            ax.pop(i)
    for i in range(len(ax) - 1, -1, -1):
        if ax[i] not in {'Y', 'X', 'S', 'C'}:
            a = np.take(a, 0, axis=i)
            ax.pop(i)
    if 'Y' not in ax or 'X' not in ax:
        raise ValueError(f'Decoded image lacks Y/X axes: {''.join(ax)}, shape={a.shape}')
    channel_axis = None
    for label in ('S', 'C'):
        if label in ax:
            channel_axis = ax.index(label)
            break
    keep_labels = {'Y', 'X'}
    if channel_axis is not None:
        keep_labels.add(ax[channel_axis])
    for i in range(len(ax) - 1, -1, -1):
        if ax[i] not in keep_labels:
            a = np.take(a, 0, axis=i)
            ax.pop(i)
    channel_axis = None
    for label in ('S', 'C'):
        if label in ax:
            channel_axis = ax.index(label)
            break
    y_axis, x_axis = (ax.index('Y'), ax.index('X'))
    if channel_axis is None:
        gray = np.moveaxis(a, (y_axis, x_axis), (0, 1))
        gray = np.asarray(gray)
        if gray.ndim > 2:
            gray = gray[..., 0]
        rgb = np.repeat(gray[..., None], 3, axis=2)
    else:
        rgb = np.moveaxis(a, (y_axis, x_axis, channel_axis), (0, 1, 2))
        if rgb.shape[2] == 1:
            rgb = np.repeat(rgb, 3, axis=2)
        elif rgb.shape[2] == 2:
            rgb = np.concatenate([rgb, rgb[..., :1]], axis=2)
        elif rgb.shape[2] > 3:
            rgb = rgb[..., :3]
    if rgb.dtype == np.uint8:
        return np.ascontiguousarray(rgb)
    if np.issubdtype(rgb.dtype, np.integer):
        max_value = np.iinfo(rgb.dtype).max
        rgb = np.clip(rgb.astype(np.float32) / max(max_value, 1) * 255.0, 0, 255)
    else:
        rgb = rgb.astype(np.float32)
        finite = np.isfinite(rgb)
        if not finite.any():
            raise ValueError('Image contains no finite pixels')
        lo, hi = np.percentile(rgb[finite], [0.5, 99.5])
        if hi <= lo:
            hi = lo + 1.0
        rgb = np.clip((rgb - lo) / (hi - lo) * 255.0, 0, 255)
    return np.ascontiguousarray(rgb.astype(np.uint8))

def read_registration_image(path: Path, cfg: Config=CFG) -> tuple[np.ndarray, dict[str, Any]]:
    meta = inspect_qptiff(path)
    level_index = _choose_level(meta['levels'], cfg.PYRAMID_TARGET_LONG_EDGE)
    with tifffile.TiffFile(path) as tif:
        series = tif.series[meta['series_index']]
        levels = list(getattr(series, 'levels', [series]))
        level = levels[level_index]
        arr = level.asarray()
        rgb = _array_to_rgb(arr, level.axes)
    original_level_shape = rgb.shape[:2]
    long_edge = max(rgb.shape[:2])
    if long_edge > cfg.PYRAMID_MAX_LONG_EDGE:
        scale = cfg.PYRAMID_MAX_LONG_EDGE / long_edge
        new_w = max(32, int(round(rgb.shape[1] * scale)))
        new_h = max(32, int(round(rgb.shape[0] * scale)))
        rgb = cv2.resize(rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    h, w = rgb.shape[:2]
    full_h, full_w = (meta['full_height'], meta['full_width'])
    low_to_full = np.array([[full_w / w, 0.0, 0.0], [0.0, full_h / h, 0.0], [0.0, 0.0, 1.0]], dtype=np.float64)
    meta.update({'selected_level': level_index, 'selected_level_shape_before_resize': tuple((int(x) for x in original_level_shape)), 'registration_height': h, 'registration_width': w, 'low_to_full': low_to_full, 'full_to_low': np.linalg.inv(low_to_full)})
    return (rgb, meta)


## Slide inventory


In [ ]:
for label, directory in [('H&E', HE_DIR), ('mIF', MIF_DIR)]:
    if not directory.exists():
        raise FileNotFoundError(f'{label} input directory does not exist: {directory}')
he_files = sorted([p for p in HE_DIR.iterdir() if p.is_file() and p.suffix.lower() in CFG.EXTENSIONS], key=lambda p: p.name.lower())
mif_files = sorted([p for p in MIF_DIR.rglob('*') if p.is_file() and p.suffix.lower() in CFG.EXTENSIONS], key=lambda p: str(p).lower())
if not he_files:
    raise RuntimeError(f'In {HE_DIR}: no H&E QPTIFF files found')
if not mif_files:
    raise RuntimeError(f'In {MIF_DIR}: no mIF QPTIFF files found')

def extract_case_id(path: Path) -> str:
    match = re.match(CFG.CASE_ID_REGEX, path.stem, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f"Filename does not match '<case_id>_ScanN.qptiff': {path.name}")
    return match.group('case_id')
inventory_rows = []
for modality, root, files in [('HE_fixed', HE_DIR, he_files), ('mIF_moving', MIF_DIR, mif_files)]:
    for path in tqdm(files, desc=f'Read {modality} metadata'):
        info = inspect_qptiff(path)
        level_index = _choose_level(info['levels'], CFG.PYRAMID_TARGET_LONG_EDGE)
        level = info['levels'][level_index]
        inventory_rows.append({'case_id': extract_case_id(path), 'modality': modality, 'file': path.name, 'relative_path': str(path.relative_to(root)), 'full_height': info['full_height'], 'full_width': info['full_width'], 'n_levels': len(info['levels']), 'selected_level': level_index, 'selected_height': level['height'], 'selected_width': level['width']})
inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(OUTPUT_DIR / 'qptiff_inventory.csv', index=False, encoding='utf-8-sig')
display(inventory_df)


## Case pairing


In [ ]:
def unique_case_map(files: list[Path], modality: str) -> dict[str, Path]:
    grouped: dict[str, list[Path]] = {}
    for path in files:
        grouped.setdefault(extract_case_id(path), []).append(path)
    duplicates = {case_id: paths for case_id, paths in grouped.items() if len(paths) != 1}
    if duplicates:
        details = '\n'.join((f'{case_id}: {[str(x) for x in paths]}' for case_id, paths in duplicates.items()))
        raise RuntimeError(f'{modality} contains duplicate case identifiers: {details}')
    return {case_id: paths[0] for case_id, paths in grouped.items()}
he_by_case = unique_case_map(he_files, 'H&E')
mif_by_case = unique_case_map(mif_files, 'mIF')
he_ids, mif_ids = (set(he_by_case), set(mif_by_case))
only_he = sorted(he_ids - mif_ids)
only_mif = sorted(mif_ids - he_ids)
if only_he or only_mif:
    raise RuntimeError(f'H&E and mIF case identifiers do not match. H&E only: {only_he}; mIF only: {only_mif}')
case_ids = sorted(he_ids & mif_ids)
if CFG.RUN_CASE_IDS:
    unknown = sorted(set(CFG.RUN_CASE_IDS) - set(case_ids))
    if unknown:
        raise ValueError(f'Unknown RUN_CASE_IDS: {unknown}')
    case_ids = [case_id for case_id in case_ids if case_id in set(CFG.RUN_CASE_IDS)]
pairs = [(he_by_case[case_id], mif_by_case[case_id]) for case_id in case_ids]
for fixed_path, moving_path in pairs:
    assert extract_case_id(fixed_path) == extract_case_id(moving_path), f'Cross-case pair: {moving_path.name} → {fixed_path.name}'
pair_df = pd.DataFrame([{'case_id': extract_case_id(fixed_path), 'fixed_modality': 'H&E', 'fixed': str(fixed_path), 'moving_modality': 'mIF', 'moving': str(moving_path), 'verified_same_case': extract_case_id(fixed_path) == extract_case_id(moving_path)} for fixed_path, moving_path in pairs])
pair_df.to_csv(OUTPUT_DIR / 'registration_pairs.csv', index=False, encoding='utf-8-sig')
display(pair_df)
if not pair_df['verified_same_case'].all():
    raise RuntimeError('Case-pairing audit failed; stopped.')


## Structural views and model


In [ ]:
processor = AutoImageProcessor.from_pretrained(CFG.MODEL_ID)
matcher = AutoModelForKeypointMatching.from_pretrained(CFG.MODEL_ID).to(DEVICE).eval()

def clean_binary_mask(mask: np.ndarray) -> np.ndarray:
    mask8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask8 = cv2.morphologyEx(mask8, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask8 = cv2.morphologyEx(mask8, cv2.MORPH_OPEN, kernel, iterations=1)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask8, connectivity=8)
    keep = np.zeros_like(mask8)
    min_area = max(32, int(mask8.size * 5e-05))
    for index in range(1, n):
        if stats[index, cv2.CC_STAT_AREA] >= min_area:
            keep[labels == index] = 255
    return keep.astype(bool)

def tissue_mask(rgb: np.ndarray, modality: str) -> np.ndarray:
    rgb = np.asarray(rgb, np.uint8)
    if modality == 'HE':
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        mask = (gray < 242) & ((hsv[..., 1] > 10) | (gray < 220))
    elif modality == 'mIF':
        signal = rgb.max(axis=2)
        otsu, _ = cv2.threshold(signal, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        threshold = max(4.0, 0.5 * float(otsu))
        mask = signal > threshold
    else:
        raise ValueError(modality)
    return clean_binary_mask(mask)

def matching_view(rgb: np.ndarray, mask: np.ndarray, modality: str) -> np.ndarray:
    if modality == 'HE':
        signal = 255 - cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    elif modality == 'mIF':
        signal = rgb.max(axis=2)
    else:
        raise ValueError(modality)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(signal.astype(np.uint8))
    view = 255 - enhanced
    view[~mask] = 255
    return np.repeat(view[..., None], 3, axis=2)

def letterbox_rgb(image: np.ndarray, canvas: int) -> tuple[np.ndarray, np.ndarray]:
    h, w = image.shape[:2]
    scale = min(canvas / w, canvas / h)
    new_w = max(1, min(canvas, int(round(w * scale))))
    new_h = max(1, min(canvas, int(round(h * scale))))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
    x0, y0 = ((canvas - new_w) // 2, (canvas - new_h) // 2)
    out = np.full((canvas, canvas, 3), 255, np.uint8)
    out[y0:y0 + new_h, x0:x0 + new_w] = resized
    matrix = np.array([[new_w / w, 0.0, x0], [0.0, new_h / h, y0], [0.0, 0.0, 1.0]], dtype=np.float64)
    return (out, matrix)

def transform_points(matrix: np.ndarray, points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, np.float64).reshape(-1, 2)
    hom = np.c_[points, np.ones(len(points))]
    warped = (np.asarray(matrix, np.float64) @ hom.T).T
    out = np.full((len(points), 2), np.nan, np.float64)
    valid = np.abs(warped[:, 2]) > 1e-12
    out[valid] = warped[valid, :2] / warped[valid, 2:3]
    return out

def orient_image(image: np.ndarray, orientation: str) -> tuple[np.ndarray, np.ndarray]:
    h, w = image.shape[:2]
    if orientation == 'identity':
        return (image.copy(), np.eye(3, dtype=np.float64))
    if orientation == 'rot90':
        matrix = np.array([[0, -1, h - 1], [1, 0, 0], [0, 0, 1]], np.float64)
        return (cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE), matrix)
    if orientation == 'rot180':
        matrix = np.array([[-1, 0, w - 1], [0, -1, h - 1], [0, 0, 1]], np.float64)
        return (cv2.rotate(image, cv2.ROTATE_180), matrix)
    if orientation == 'rot270':
        matrix = np.array([[0, 1, 0], [-1, 0, w - 1], [0, 0, 1]], np.float64)
        return (cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE), matrix)
    if orientation == 'flip_h':
        matrix = np.array([[-1, 0, w - 1], [0, 1, 0], [0, 0, 1]], np.float64)
        return (cv2.flip(image, 1), matrix)
    if orientation == 'flip_v':
        matrix = np.array([[1, 0, 0], [0, -1, h - 1], [0, 0, 1]], np.float64)
        return (cv2.flip(image, 0), matrix)
    raise ValueError(f'Unknown orientation: {orientation}')

def points_inside_mask(mask: np.ndarray, points: np.ndarray) -> np.ndarray:
    xy = np.round(points).astype(np.int64)
    h, w = mask.shape
    valid = np.isfinite(points).all(axis=1) & (xy[:, 0] >= 0) & (xy[:, 0] < w) & (xy[:, 1] >= 0) & (xy[:, 1] < h)
    inside = np.zeros(len(points), dtype=bool)
    indices = np.where(valid)[0]
    inside[indices] = mask[xy[indices, 1], xy[indices, 0]]
    return inside

def infer_correspondences(image0: np.ndarray, image1: np.ndarray) -> dict[str, Any]:
    canvas0, to_canvas0 = letterbox_rgb(image0, CFG.MODEL_CANVAS)
    canvas1, to_canvas1 = letterbox_rgb(image1, CFG.MODEL_CANVAS)
    inputs = processor([Image.fromarray(canvas0), Image.fromarray(canvas1)], do_resize=False, return_tensors='pt')
    inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}
    amp_enabled = CFG.USE_MIXED_PRECISION and DEVICE.type == 'cuda'
    amp_context = torch.autocast('cuda', dtype=torch.float16) if amp_enabled else nullcontext()
    with torch.inference_mode(), amp_context:
        outputs = matcher(**inputs)
    result = processor.post_process_keypoint_matching(outputs, target_sizes=[[(CFG.MODEL_CANVAS, CFG.MODEL_CANVAS), (CFG.MODEL_CANVAS, CFG.MODEL_CANVAS)]], threshold=CFG.MATCH_MIN_SCORE)[0]
    pts0_canvas = result['keypoints0'].detach().cpu().numpy().astype(np.float64)
    pts1_canvas = result['keypoints1'].detach().cpu().numpy().astype(np.float64)
    scores = result['matching_scores'].detach().cpu().numpy().astype(np.float64)
    return {'pts0': transform_points(np.linalg.inv(to_canvas0), pts0_canvas), 'pts1': transform_points(np.linalg.inv(to_canvas1), pts1_canvas), 'scores': scores, 'n_model_candidates': int(len(scores))}

def normalized_pair_descriptor(fixed_pts: np.ndarray, moving_pts: np.ndarray, fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> np.ndarray:
    hf, wf = fixed_shape
    hm, wm = moving_shape
    return np.column_stack([fixed_pts[:, 0] / max(wf - 1, 1) * CFG.MODEL_CANVAS, fixed_pts[:, 1] / max(hf - 1, 1) * CFG.MODEL_CANVAS, moving_pts[:, 0] / max(wm - 1, 1) * CFG.MODEL_CANVAS, moving_pts[:, 1] / max(hm - 1, 1) * CFG.MODEL_CANVAS]).astype(np.float32)

def mutual_consistency_filter(f_fixed: np.ndarray, f_moving: np.ndarray, f_scores: np.ndarray, r_fixed: np.ndarray, r_moving: np.ndarray, r_scores: np.ndarray, fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not len(f_scores) or not len(r_scores):
        return (f_fixed[:0], f_moving[:0], f_scores[:0])
    desc_f = normalized_pair_descriptor(f_fixed, f_moving, fixed_shape, moving_shape)
    desc_r = normalized_pair_descriptor(r_fixed, r_moving, fixed_shape, moving_shape)
    matcher4d = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    mutual = matcher4d.match(desc_f, desc_r)
    accepted = [m for m in mutual if m.distance <= CFG.MUTUAL_TOLERANCE_CANVAS_PX]
    if not accepted:
        return (f_fixed[:0], f_moving[:0], f_scores[:0])
    q = np.asarray([m.queryIdx for m in accepted], dtype=np.int64)
    t = np.asarray([m.trainIdx for m in accepted], dtype=np.int64)
    scores = np.sqrt(np.clip(f_scores[q], 0, 1) * np.clip(r_scores[t], 0, 1))
    return (f_fixed[q], f_moving[q], scores)

def match_orientation(fixed_view: np.ndarray, moving_view: np.ndarray, fixed_mask: np.ndarray, moving_mask: np.ndarray, orientation: str) -> dict[str, Any]:
    oriented_moving, moving_to_oriented = orient_image(moving_view, orientation)
    oriented_mask, _ = orient_image(moving_mask.astype(np.uint8), orientation)
    oriented_mask = oriented_mask.astype(bool)
    forward = infer_correspondences(fixed_view, oriented_moving)
    f_fixed = forward['pts0']
    f_oriented = forward['pts1']
    f_moving = transform_points(np.linalg.inv(moving_to_oriented), f_oriented)
    valid_f = points_inside_mask(fixed_mask, f_fixed) & points_inside_mask(moving_mask, f_moving)
    f_fixed, f_moving, f_scores = (f_fixed[valid_f], f_moving[valid_f], forward['scores'][valid_f])
    reverse_count = 0
    mutual_used = False
    n_after_mutual = 0
    if CFG.USE_BIDIRECTIONAL_MATCHING:
        reverse = infer_correspondences(oriented_moving, fixed_view)
        reverse_count = reverse['n_model_candidates']
        r_oriented = reverse['pts0']
        r_fixed = reverse['pts1']
        r_moving = transform_points(np.linalg.inv(moving_to_oriented), r_oriented)
        valid_r = points_inside_mask(fixed_mask, r_fixed) & points_inside_mask(moving_mask, r_moving)
        r_fixed, r_moving, r_scores = (r_fixed[valid_r], r_moving[valid_r], reverse['scores'][valid_r])
        m_fixed, m_moving, m_scores = mutual_consistency_filter(f_fixed, f_moving, f_scores, r_fixed, r_moving, r_scores, fixed_view.shape[:2], moving_view.shape[:2])
        n_after_mutual = int(len(m_scores))
        if n_after_mutual >= CFG.MIN_MUTUAL_MATCHES:
            f_fixed, f_moving, f_scores = (m_fixed, m_moving, m_scores)
            mutual_used = True
    return {'orientation': orientation, 'fixed_pts': f_fixed, 'moving_pts': f_moving, 'scores': f_scores, 'n_model_candidates': forward['n_model_candidates'], 'n_reverse_model_candidates': reverse_count, 'n_after_tissue_filter': int(valid_f.sum()), 'n_after_mutual_filter': n_after_mutual, 'mutual_used': mutual_used, 'moving_to_oriented': moving_to_oriented}


## Robust global matching


In [ ]:
def spatial_coverage(points: np.ndarray, image_shape: tuple[int, int]) -> float:
    if len(points) < 3:
        return 0.0
    hull = cv2.convexHull(np.asarray(points, np.float32))
    h, w = image_shape
    return float(cv2.contourArea(hull)) / max(float(h * w), 1.0)

def model_metrics(matrix: Optional[np.ndarray], inliers: Optional[np.ndarray], moving_pts: np.ndarray, fixed_pts: np.ndarray, moving_shape: tuple[int, int], fixed_shape: tuple[int, int]) -> dict[str, Any]:
    if matrix is None or inliers is None:
        return {'valid': False, 'n_inliers': 0, 'inlier_ratio': 0.0}
    mask = np.asarray(inliers).reshape(-1).astype(bool)
    projected = transform_points(matrix, moving_pts)
    errors = np.linalg.norm(projected - fixed_pts, axis=1)
    accepted = errors[mask]
    return {'valid': True, 'n_candidate_matches': int(len(mask)), 'n_inliers': int(mask.sum()), 'n_outliers': int(len(mask) - mask.sum()), 'inlier_ratio': float(mask.mean()) if len(mask) else 0.0, 'rmse_px': float(np.sqrt(np.mean(accepted ** 2))) if len(accepted) else None, 'median_error_px': float(np.median(accepted)) if len(accepted) else None, 'p95_error_px': float(np.percentile(accepted, 95)) if len(accepted) else None, 'moving_coverage': spatial_coverage(moving_pts[mask], moving_shape), 'fixed_coverage': spatial_coverage(fixed_pts[mask], fixed_shape)}

def geometry_diagnostics(matrix: np.ndarray, fixed_shape: tuple[int, int]) -> dict[str, Any]:
    linear = np.asarray(matrix, np.float64)[:2, :2]
    singular = np.linalg.svd(linear, compute_uv=False)
    scale_max, scale_min = (float(singular.max()), float(singular.min()))
    determinant = float(np.linalg.det(linear))
    anisotropy = scale_max / max(scale_min, 1e-12)
    rotation_deg = float(np.degrees(np.arctan2(linear[1, 0], linear[0, 0])))
    translation = np.asarray(matrix, np.float64)[:2, 2]
    h, w = fixed_shape
    translation_fraction = float(np.linalg.norm(translation) / max(math.hypot(w, h), 1.0))
    plausible = scale_min >= CFG.MIN_SCALE and scale_max <= CFG.MAX_SCALE and (anisotropy <= CFG.MAX_ANISOTROPY) and (translation_fraction <= CFG.MAX_TRANSLATION_FRACTION) and (abs(determinant) > 1e-08) and (CFG.ALLOW_REFLECTION or determinant > 0)
    return {'scale_min': scale_min, 'scale_max': scale_max, 'anisotropy': anisotropy, 'determinant': determinant, 'reflection': determinant < 0, 'rotation_deg': rotation_deg, 'translation_fraction': translation_fraction, 'plausible': bool(plausible)}

def estimate_one_model(model_name: str, moving_pts: np.ndarray, fixed_pts: np.ndarray) -> tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    if len(moving_pts) < 3:
        return (None, None)
    if model_name == 'similarity':
        affine2, mask = cv2.estimateAffinePartial2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=cv2.RANSAC, ransacReprojThreshold=CFG.RANSAC_THRESHOLD_PX, maxIters=CFG.RANSAC_MAX_ITERS, confidence=CFG.RANSAC_CONFIDENCE, refineIters=25)
    elif model_name == 'affine':
        method = getattr(cv2, 'USAC_MAGSAC', cv2.RANSAC)
        try:
            affine2, mask = cv2.estimateAffine2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=method, ransacReprojThreshold=CFG.RANSAC_THRESHOLD_PX, maxIters=CFG.RANSAC_MAX_ITERS, confidence=CFG.RANSAC_CONFIDENCE, refineIters=25)
        except cv2.error:
            affine2, mask = cv2.estimateAffine2D(moving_pts.astype(np.float32), fixed_pts.astype(np.float32), method=cv2.RANSAC, ransacReprojThreshold=CFG.RANSAC_THRESHOLD_PX, maxIters=CFG.RANSAC_MAX_ITERS, confidence=CFG.RANSAC_CONFIDENCE, refineIters=25)
    else:
        raise ValueError(model_name)
    if affine2 is None:
        return (None, None)
    matrix = np.vstack([affine2, [0.0, 0.0, 1.0]]).astype(np.float64)
    return (matrix, None if mask is None else mask.reshape(-1).astype(bool))

def grid_balance_indices(fixed_pts: np.ndarray, moving_pts: np.ndarray, scores: np.ndarray, fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> np.ndarray:
    if not len(scores):
        return np.empty(0, dtype=np.int64)
    hf, wf = fixed_shape
    hm, wm = moving_shape
    gx_f = np.clip((fixed_pts[:, 0] / max(wf, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    gy_f = np.clip((fixed_pts[:, 1] / max(hf, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    gx_m = np.clip((moving_pts[:, 0] / max(wm, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    gy_m = np.clip((moving_pts[:, 1] / max(hm, 1) * CFG.GRID_SIZE).astype(int), 0, CFG.GRID_SIZE - 1)
    counts_f: dict[tuple[int, int], int] = {}
    counts_m: dict[tuple[int, int], int] = {}
    selected = []
    for i in np.argsort(scores)[::-1]:
        cf, cm = ((int(gx_f[i]), int(gy_f[i])), (int(gx_m[i]), int(gy_m[i])))
        if counts_f.get(cf, 0) >= CFG.MAX_MATCHES_PER_GRID_CELL:
            continue
        if counts_m.get(cm, 0) >= CFG.MAX_MATCHES_PER_GRID_CELL:
            continue
        selected.append(int(i))
        counts_f[cf] = counts_f.get(cf, 0) + 1
        counts_m[cm] = counts_m.get(cm, 0) + 1
        if len(selected) >= CFG.MAX_RANSAC_CANDIDATES:
            break
    return np.asarray(selected, dtype=np.int64)

def candidate_quality(candidate: dict[str, Any]) -> float:
    metrics, geometry = (candidate['metrics'], candidate['geometry'])
    if not metrics.get('valid') or not geometry.get('plausible'):
        return -1.0
    n = metrics['n_inliers']
    ratio = metrics['inlier_ratio']
    coverage = min(metrics['fixed_coverage'], metrics['moving_coverage'])
    median = metrics.get('median_error_px')
    median = CFG.RANSAC_THRESHOLD_PX if median is None else median
    score = math.log1p(n) * max(ratio, 1e-06) ** 3 * math.sqrt(0.01 + coverage)
    score /= 1.0 + median / max(CFG.RANSAC_THRESHOLD_PX, 1.0)
    if not candidate.get('mutual_used', False):
        score *= 0.2
    if candidate['model'] == 'affine':
        score *= 0.85
    return float(score)

def evaluate_match_result(match_result: dict[str, Any], fixed_shape: tuple[int, int], moving_shape: tuple[int, int]) -> list[dict[str, Any]]:
    candidates = []
    all_scores = match_result['scores']
    for threshold in CFG.SCORE_THRESHOLDS:
        keep = all_scores >= threshold
        fixed_raw = match_result['fixed_pts'][keep]
        moving_raw = match_result['moving_pts'][keep]
        scores_raw = all_scores[keep]
        if len(scores_raw) < 3:
            continue
        balanced = grid_balance_indices(fixed_raw, moving_raw, scores_raw, fixed_shape, moving_shape)
        fixed_pts, moving_pts, scores = (fixed_raw[balanced], moving_raw[balanced], scores_raw[balanced])
        if len(scores) < 3:
            continue
        model_names = ['similarity'] + (['affine'] if CFG.ALLOW_AFFINE else [])
        for model_name in model_names:
            matrix, inliers = estimate_one_model(model_name, moving_pts, fixed_pts)
            metrics = model_metrics(matrix, inliers, moving_pts, fixed_pts, moving_shape, fixed_shape)
            geometry = geometry_diagnostics(matrix, fixed_shape) if matrix is not None else {'plausible': False}
            candidate = {'orientation': match_result['orientation'], 'score_threshold': float(threshold), 'model': model_name, 'matrix': matrix, 'inliers': inliers, 'fixed_pts': fixed_pts, 'moving_pts': moving_pts, 'scores': scores, 'metrics': metrics, 'geometry': geometry, 'n_model_candidates': match_result['n_model_candidates'], 'n_reverse_model_candidates': match_result['n_reverse_model_candidates'], 'n_after_tissue_filter': match_result['n_after_tissue_filter'], 'n_after_mutual_filter': match_result['n_after_mutual_filter'], 'n_before_grid_filter': int(len(scores_raw)), 'n_after_grid_filter': int(len(scores)), 'mutual_used': match_result['mutual_used']}
            candidate['quality_score'] = candidate_quality(candidate)
            candidates.append(candidate)
    return candidates

def choose_best(candidates: list[dict[str, Any]]) -> Optional[dict[str, Any]]:
    eligible = [c for c in candidates if c['quality_score'] >= 0 and c['metrics'].get('n_inliers', 0) >= 3]
    if not eligible:
        return None
    strict = []
    for c in eligible:
        m = c['metrics']
        coverage = min(m.get('fixed_coverage', 0.0), m.get('moving_coverage', 0.0))
        if c.get('mutual_used', False) and m.get('n_inliers', 0) >= CFG.QC_MIN_INLIERS and (m.get('inlier_ratio', 0.0) >= CFG.QC_MIN_INLIER_RATIO) and (coverage >= CFG.QC_MIN_SPATIAL_COVERAGE):
            strict.append(c)
    pool = strict if strict else eligible
    return max(pool, key=lambda c: c['quality_score'])

def fast_accept(candidate: Optional[dict[str, Any]]) -> bool:
    if candidate is None:
        return False
    m = candidate['metrics']
    coverage = min(m.get('fixed_coverage', 0.0), m.get('moving_coverage', 0.0))
    return candidate.get('mutual_used', False) and candidate['geometry'].get('plausible', False) and (m.get('n_inliers', 0) >= CFG.FAST_ACCEPT_MIN_INLIERS) and (m.get('inlier_ratio', 0.0) >= CFG.FAST_ACCEPT_MIN_RATIO) and (coverage >= CFG.FAST_ACCEPT_MIN_COVERAGE)


## QC exports


In [ ]:
def safe_slug(text: str) -> str:
    return re.sub('[^0-9A-Za-z._-]+', '_', text).strip('._') or 'case'

def json_ready(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(x) for x in obj]
    return obj

def save_rgb(path: Path, rgb: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(path), cv2.cvtColor(np.asarray(rgb, np.uint8), cv2.COLOR_RGB2BGR))
    if not ok:
        raise IOError(f'Cannot save: {path}')

def warp_rgb(moving_rgb: np.ndarray, matrix: np.ndarray, fixed_shape: tuple[int, int]) -> np.ndarray:
    h, w = fixed_shape
    return cv2.warpPerspective(moving_rgb, matrix.astype(np.float64), (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

def warp_mask(mask: np.ndarray, matrix: np.ndarray, fixed_shape: tuple[int, int]) -> np.ndarray:
    h, w = fixed_shape
    warped = cv2.warpPerspective(mask.astype(np.uint8), matrix.astype(np.float64), (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return warped.astype(bool)

def make_checkerboard(a: np.ndarray, b: np.ndarray, block: int=96) -> np.ndarray:
    yy, xx = np.indices(a.shape[:2])
    choose_a = ((xx // block + yy // block) % 2).astype(bool)
    return np.where(choose_a[..., None], a, b).astype(np.uint8)

def make_match_qc(fixed_rgb: np.ndarray, moving_rgb: np.ndarray, fixed_pts: np.ndarray, moving_pts: np.ndarray, scores: np.ndarray, inliers: np.ndarray) -> np.ndarray:
    max_edge = 1100

    def resize_with_matrix(image):
        h, w = image.shape[:2]
        scale = min(1.0, max_edge / max(h, w))
        nw, nh = (max(1, round(w * scale)), max(1, round(h * scale)))
        out = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA) if scale < 1 else image.copy()
        return (out, np.array([[nw / w, 0, 0], [0, nh / h, 0], [0, 0, 1]], np.float64))
    fixed_show, sf = resize_with_matrix(fixed_rgb)
    moving_show, sm = resize_with_matrix(moving_rgb)
    hf, wf = fixed_show.shape[:2]
    hm, wm = moving_show.shape[:2]
    canvas = np.full((max(hf, hm), wf + wm, 3), 255, np.uint8)
    canvas[:hf, :wf] = fixed_show
    canvas[:hm, wf:wf + wm] = moving_show
    p_fixed = transform_points(sf, fixed_pts)
    p_moving = transform_points(sm, moving_pts)
    order = np.argsort(scores)[::-1][:CFG.MAX_QC_MATCHES]
    for wanted, color in [(False, (230, 70, 70)), (True, (40, 210, 90))]:
        for i in order:
            if bool(inliers[i]) != wanted:
                continue
            a = tuple(np.round(p_fixed[i]).astype(int))
            b0 = np.round(p_moving[i]).astype(int)
            b = (int(b0[0] + wf), int(b0[1]))
            cv2.line(canvas, a, b, color, 1, cv2.LINE_AA)
            cv2.circle(canvas, a, 2, color, -1, cv2.LINE_AA)
            cv2.circle(canvas, b, 2, color, -1, cv2.LINE_AA)
    cv2.putText(canvas, 'FIXED H&E', (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (10, 10, 10), 2)
    cv2.putText(canvas, 'MOVING mIF', (wf + 15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (10, 10, 10), 2)
    return canvas

def tissue_overlap_metrics(fixed_mask: np.ndarray, warped_moving_mask: np.ndarray) -> dict[str, float]:
    intersection = int((fixed_mask & warped_moving_mask).sum())
    fixed_area = int(fixed_mask.sum())
    moving_area = int(warped_moving_mask.sum())
    return {'tissue_dice': 2.0 * intersection / max(fixed_area + moving_area, 1), 'fixed_tissue_covered': intersection / max(fixed_area, 1), 'moving_tissue_covered': intersection / max(moving_area, 1)}

def make_tissue_contour_qc(fixed_rgb: np.ndarray, fixed_mask: np.ndarray, warped_moving_mask: np.ndarray) -> np.ndarray:
    canvas = fixed_rgb.copy()
    fixed_contours, _ = cv2.findContours(fixed_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    moving_contours, _ = cv2.findContours(warped_moving_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, fixed_contours, -1, (20, 210, 60), 3)
    cv2.drawContours(canvas, moving_contours, -1, (235, 40, 210), 3)
    return canvas

def full_resolution_matrix(low_matrix: np.ndarray, fixed_low_to_full: np.ndarray, moving_low_to_full: np.ndarray) -> np.ndarray:
    matrix = fixed_low_to_full @ low_matrix @ np.linalg.inv(moving_low_to_full)
    if abs(matrix[2, 2]) > 1e-12:
        matrix /= matrix[2, 2]
    return matrix

def candidate_table(candidates: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for c in candidates:
        m, g = (c['metrics'], c['geometry'])
        rows.append({'orientation': c['orientation'], 'score_threshold': c['score_threshold'], 'model': c['model'], 'quality_score': c['quality_score'], 'n_forward_model_candidates': c['n_model_candidates'], 'n_reverse_model_candidates': c.get('n_reverse_model_candidates', 0), 'n_after_tissue_filter': c['n_after_tissue_filter'], 'n_after_mutual_filter': c.get('n_after_mutual_filter', 0), 'mutual_used': c.get('mutual_used', False), 'n_before_grid_filter': c.get('n_before_grid_filter', 0), 'n_after_grid_filter': c.get('n_after_grid_filter', 0), 'n_candidate_matches': m.get('n_candidate_matches', 0), 'n_inliers': m.get('n_inliers', 0), 'n_outliers': m.get('n_outliers', 0), 'inlier_ratio': m.get('inlier_ratio', 0.0), 'median_error_px': m.get('median_error_px'), 'fixed_coverage': m.get('fixed_coverage', 0.0), 'moving_coverage': m.get('moving_coverage', 0.0), 'scale_min': g.get('scale_min'), 'scale_max': g.get('scale_max'), 'anisotropy': g.get('anisotropy'), 'rotation_deg': g.get('rotation_deg'), 'translation_fraction': g.get('translation_fraction'), 'plausible': g.get('plausible', False)})
    return pd.DataFrame(rows).sort_values('quality_score', ascending=False) if rows else pd.DataFrame()


## Pair registration


In [ ]:
def register_pair(fixed_path: Path, moving_path: Path, cfg: Config=CFG) -> dict[str, Any]:
    started = time.time()
    fixed_case, moving_case = (extract_case_id(fixed_path), extract_case_id(moving_path))
    if fixed_case != moving_case:
        raise ValueError(f'Cross-case registration is prohibited: mIF {moving_case} → H&E {fixed_case}')
    case_id = fixed_case
    pair_dir = OUTPUT_DIR / 'pairs' / f'{safe_slug(case_id)}__mIF_to_HE'
    pair_dir.mkdir(parents=True, exist_ok=True)
    fixed_rgb, fixed_meta = read_registration_image(fixed_path, cfg)
    moving_rgb, moving_meta = read_registration_image(moving_path, cfg)
    fixed_mask, moving_mask = (tissue_mask(fixed_rgb, 'HE'), tissue_mask(moving_rgb, 'mIF'))
    fixed_view = matching_view(fixed_rgb, fixed_mask, 'HE')
    moving_view = matching_view(moving_rgb, moving_mask, 'mIF')
    save_rgb(pair_dir / 'fixed_registration.png', fixed_rgb)
    save_rgb(pair_dir / 'moving_registration.png', moving_rgb)
    save_rgb(pair_dir / 'fixed_matching_preprocessed.png', fixed_view)
    save_rgb(pair_dir / 'moving_matching_preprocessed.png', moving_view)
    save_rgb(pair_dir / 'fixed_tissue_mask.png', np.repeat((fixed_mask * 255)[..., None], 3, axis=2))
    save_rgb(pair_dir / 'moving_tissue_mask.png', np.repeat((moving_mask * 255)[..., None], 3, axis=2))
    all_candidates: list[dict[str, Any]] = []
    identity_match = match_orientation(fixed_view, moving_view, fixed_mask, moving_mask, 'identity')
    all_candidates.extend(evaluate_match_result(identity_match, fixed_rgb.shape[:2], moving_rgb.shape[:2]))
    best = choose_best(all_candidates)
    attempted_orientations = ['identity']
    if cfg.ENABLE_ORIENTATION_RESCUE and (not fast_accept(best)):
        for orientation in cfg.RESCUE_ORIENTATIONS:
            attempted_orientations.append(orientation)
            result = match_orientation(fixed_view, moving_view, fixed_mask, moving_mask, orientation)
            all_candidates.extend(evaluate_match_result(result, fixed_rgb.shape[:2], moving_rgb.shape[:2]))
        best = choose_best(all_candidates)
    candidate_table(all_candidates).to_csv(pair_dir / 'candidate_search.csv', index=False, encoding='utf-8-sig')
    if best is None:
        raise RuntimeError('No orientation or threshold yields a geometrically plausible transform')
    metrics, geometry = (best['metrics'], best['geometry'])
    if metrics['n_inliers'] < cfg.HARD_MIN_INLIERS:
        raise RuntimeError(f'Best candidate has only {metrics['n_inliers']} inliers')
    if not geometry['plausible']:
        raise RuntimeError('Best transform fails scale, shear or translation checks')
    low_matrix = best['matrix']
    full_matrix = full_resolution_matrix(low_matrix, fixed_meta['low_to_full'], moving_meta['low_to_full'])
    np.save(pair_dir / 'transform_candidate_registration.npy', low_matrix)
    np.save(pair_dir / 'transform_candidate_full.npy', full_matrix)
    matches_df = pd.DataFrame({'score': best['scores'], 'fixed_x_registration': best['fixed_pts'][:, 0], 'fixed_y_registration': best['fixed_pts'][:, 1], 'moving_x_registration': best['moving_pts'][:, 0], 'moving_y_registration': best['moving_pts'][:, 1], 'inlier': best['inliers']})
    fixed_full = transform_points(fixed_meta['low_to_full'], best['fixed_pts'])
    moving_full = transform_points(moving_meta['low_to_full'], best['moving_pts'])
    matches_df['fixed_x_full'], matches_df['fixed_y_full'] = (fixed_full[:, 0], fixed_full[:, 1])
    matches_df['moving_x_full'], matches_df['moving_y_full'] = (moving_full[:, 0], moving_full[:, 1])
    matches_df.to_csv(pair_dir / 'matches_selected.csv', index=False, encoding='utf-8-sig')
    warped = warp_rgb(moving_rgb, low_matrix, fixed_rgb.shape[:2])
    warped_mask = warp_mask(moving_mask, low_matrix, fixed_rgb.shape[:2])
    tissue_metrics = tissue_overlap_metrics(fixed_mask, warped_mask)
    identity_mask = warp_mask(moving_mask, np.eye(3), fixed_rgb.shape[:2])
    identity_tissue = tissue_overlap_metrics(fixed_mask, identity_mask)
    tissue_metrics['tissue_dice_before'] = identity_tissue['tissue_dice']
    tissue_metrics['tissue_dice_gain'] = tissue_metrics['tissue_dice'] - identity_tissue['tissue_dice']
    save_rgb(pair_dir / 'moving_warped_registration.png', warped)
    save_rgb(pair_dir / 'qc_overlay.png', cv2.addWeighted(fixed_rgb, 0.55, warped, 0.45, 0.0))
    save_rgb(pair_dir / 'qc_checkerboard.png', make_checkerboard(fixed_rgb, warped))
    save_rgb(pair_dir / 'qc_matches.png', make_match_qc(fixed_rgb, moving_rgb, best['fixed_pts'], best['moving_pts'], best['scores'], best['inliers']))
    save_rgb(pair_dir / 'qc_tissue_contours.png', make_tissue_contour_qc(fixed_rgb, fixed_mask, warped_mask))
    min_coverage = min(metrics['fixed_coverage'], metrics['moving_coverage'])
    warnings = []
    if not best.get('mutual_used', False):
        warnings.append('bidirectional_mutual_filter_unavailable')
    if metrics['n_candidate_matches'] < cfg.QC_MIN_CANDIDATE_MATCHES:
        warnings.append(f'candidate_matches<{cfg.QC_MIN_CANDIDATE_MATCHES}')
    if metrics['n_inliers'] < cfg.QC_MIN_INLIERS:
        warnings.append(f'inliers<{cfg.QC_MIN_INLIERS}')
    if metrics['inlier_ratio'] < cfg.QC_MIN_INLIER_RATIO:
        warnings.append(f'inlier_ratio<{cfg.QC_MIN_INLIER_RATIO}')
    if metrics['n_outliers'] > metrics['n_inliers']:
        warnings.append('outliers_exceed_inliers')
    if min_coverage < cfg.QC_MIN_SPATIAL_COVERAGE:
        warnings.append(f'spatial_coverage<{cfg.QC_MIN_SPATIAL_COVERAGE}')
    if (metrics.get('median_error_px') or float('inf')) > cfg.QC_MAX_MEDIAN_ERROR_PX:
        warnings.append(f'median_error>{cfg.QC_MAX_MEDIAN_ERROR_PX}px')
    if tissue_metrics['tissue_dice'] < cfg.QC_MIN_TISSUE_DICE:
        warnings.append(f'tissue_dice<{cfg.QC_MIN_TISSUE_DICE}')
    hard_fail = metrics['n_inliers'] < cfg.HARD_MIN_INLIERS or metrics['inlier_ratio'] < cfg.FAIL_MIN_INLIER_RATIO
    status = 'failed' if hard_fail else 'warning' if warnings else 'ok'
    if status == 'ok':
        np.save(pair_dir / 'transform_registration.npy', low_matrix)
        np.save(pair_dir / 'transform_full.npy', full_matrix)
    report = {'case_id': case_id, 'status': status, 'accepted_for_downstream': status == 'ok', 'warnings': warnings, 'pairing_verified': True, 'direction': 'mIF moving full-resolution -> H&E fixed full-resolution', 'fixed_he': str(fixed_path), 'moving_mif': str(moving_path), 'selected_orientation': best['orientation'], 'attempted_orientations': attempted_orientations, 'selected_score_threshold': best['score_threshold'], 'selected_model': best['model'], 'quality_score': best['quality_score'], 'filter_counts': {'forward_model_candidates': best['n_model_candidates'], 'reverse_model_candidates': best.get('n_reverse_model_candidates', 0), 'after_tissue_filter': best['n_after_tissue_filter'], 'after_mutual_filter': best.get('n_after_mutual_filter', 0), 'mutual_used': best.get('mutual_used', False), 'before_grid_filter': best.get('n_before_grid_filter', 0), 'after_grid_filter_ransac_candidates': best.get('n_after_grid_filter', 0)}, 'selected_metrics': metrics, 'geometry': geometry, 'tissue_overlap': tissue_metrics, 'transform_candidate_registration': low_matrix, 'transform_candidate_full': full_matrix, 'fixed_metadata': fixed_meta, 'moving_metadata': moving_meta, 'elapsed_seconds': time.time() - started}
    with (pair_dir / 'metrics.json').open('w', encoding='utf-8') as f:
        json.dump(json_ready(report), f, ensure_ascii=False, indent=2)
    return {'case_id': case_id, 'status': status, 'accepted_for_downstream': status == 'ok', 'warnings': ';'.join(warnings), 'fixed': fixed_path.name, 'moving': moving_path.name, 'pairing_verified': True, 'orientation': best['orientation'], 'score_threshold': best['score_threshold'], 'selected_model': best['model'], 'quality_score': best['quality_score'], 'n_forward_raw': best['n_model_candidates'], 'n_after_tissue': best['n_after_tissue_filter'], 'n_after_mutual': best.get('n_after_mutual_filter', 0), 'mutual_used': best.get('mutual_used', False), 'n_candidate_matches': metrics['n_candidate_matches'], 'n_inliers': metrics['n_inliers'], 'n_outliers': metrics['n_outliers'], 'inlier_ratio': metrics['inlier_ratio'], 'median_error_px_registration': metrics.get('median_error_px'), 'fixed_coverage': metrics['fixed_coverage'], 'moving_coverage': metrics['moving_coverage'], 'tissue_dice': tissue_metrics['tissue_dice'], 'tissue_dice_gain': tissue_metrics['tissue_dice_gain'], 'scale_min': geometry['scale_min'], 'scale_max': geometry['scale_max'], 'anisotropy': geometry['anisotropy'], 'elapsed_seconds': report['elapsed_seconds'], 'output_dir': str(pair_dir)}


## Run


In [ ]:
run_info = {'config': asdict(CFG), 'he_fixed_dir': str(HE_DIR), 'mif_moving_dir': str(MIF_DIR), 'output_dir': str(OUTPUT_DIR), 'quality_policy': 'ok requires mutual matching and inliers >= outliers (ratio >= 0.50)', 'device': str(DEVICE), 'torch_version': torch.__version__, 'transformers_version': transformers.__version__, 'opencv_version': cv2.__version__, 'pairing_rule': 'strict same case_id before _Scan; mIF moving -> H&E fixed', 'pairs': [{'case_id': extract_case_id(fixed), 'fixed_he': str(fixed), 'moving_mif': str(moving), 'verified_same_case': extract_case_id(fixed) == extract_case_id(moving)} for fixed, moving in pairs]}
if not all((x['verified_same_case'] for x in run_info['pairs'])):
    raise RuntimeError('Case-pairing audit failed; inference is stopped.')
with (OUTPUT_DIR / 'run_config.json').open('w', encoding='utf-8') as f:
    json.dump(json_ready(run_info), f, ensure_ascii=False, indent=2)
summary_rows = []
for fixed_path, moving_path in tqdm(pairs, desc='Robust mIF -> H&E registration'):
    case_id = extract_case_id(fixed_path)
    try:
        row = register_pair(fixed_path, moving_path, CFG)
    except Exception as exc:
        pair_dir = OUTPUT_DIR / 'pairs' / f'{safe_slug(case_id)}__mIF_to_HE'
        pair_dir.mkdir(parents=True, exist_ok=True)
        error_report = {'case_id': case_id, 'status': 'failed', 'fixed_he': str(fixed_path), 'moving_mif': str(moving_path), 'pairing_verified': extract_case_id(fixed_path) == extract_case_id(moving_path), 'error_type': type(exc).__name__, 'error': str(exc), 'traceback': traceback.format_exc()}
        with (pair_dir / 'error.json').open('w', encoding='utf-8') as f:
            json.dump(error_report, f, ensure_ascii=False, indent=2)
        row = {'case_id': case_id, 'status': 'failed', 'warnings': '', 'fixed': fixed_path.name, 'moving': moving_path.name, 'pairing_verified': True, 'output_dir': str(pair_dir), 'error': str(exc)}
    summary_rows.append(row)
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'registration_summary.csv', index=False, encoding='utf-8-sig')
display(summary_df)
successful = summary_df[summary_df['status'].isin(['ok', 'warning'])].copy()
if len(successful):
    total_candidates = int(successful['n_candidate_matches'].sum())
    total_inliers = int(successful['n_inliers'].sum())
